# 02 — Integration of the circulating cells

The datasets come from different labs and technologies. This notebook puts the circulating hemocytes of all
datasets into one shared space in which cells of the same type lie together, whatever dataset they come from.

Steps:
1. Choose 2,000 highly variable genes (HVGs), using only the unperturbed control cells of all eight datasets.
2. Keep the circulating cells and reduce them to 15 principal components (PCA).
3. Remove batch differences with Harmony.
4. Compute the UMAP layout that the figures use.

Outputs: the HVG table (`integration_hvgs.tsv`) and `integration_circulating.h5ad` (Harmony coordinates and UMAP).

In [ ]:
import anndata as ad
import harmonypy
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

from settings import ATLAS_FILE, HVG_TABLE, INPUT_DIR, INTEGRATION_FILE, SEED

## Settings

In [ ]:
# highly variable genes
EXCLUDED_GENES = ["Mst84Da", "Mst84Db", "Mst87F", "S-Lap7", "S-Lap8"]      # male germline genes
MIN_CONTROL_CELLS = 100           # a gene must be detected in at least this many control cells ...
MIN_CELLS_PER_DATASET = 5         # ... and in at least this many control cells ...
MIN_DATASETS = 4                  # ... of at least this many datasets
HVGS_PER_DATASET = 4_000          # HVGs called in each dataset
N_HVGS = 2_000                    # genes kept for the integration

# PCA, Harmony, UMAP
N_PCS = 15
SCALE_MAX_VALUE = 10
HARMONY_THETA = 2
HARMONY_MAX_ITERATIONS = 20
N_NEIGHBORS = 20
UMAP_MIN_DIST = 0.3

UMAP_START_POSITIONS = INPUT_DIR / "umap_start_positions.tsv.gz"

In [ ]:
atlas = ad.read_h5ad(ATLAS_FILE)
print(atlas)

## 1. Highly variable genes

Only **unperturbed control cells** are used to choose the genes (`condition == "control"`: all datasets, all
compartments, including the control arm of the AML dataset). This keeps the choice free of genes that vary only
because of infection, injury or the leukemia transgenes.

First the genes that may be chosen at all ("eligible"):
- not mitochondrial, not a transgene, not one of the male germline genes;
- detected in ≥ 100 control cells;
- detected in ≥ 5 control cells in each of ≥ 4 datasets.

In [ ]:
is_control = atlas.obs["condition"].eq("control").to_numpy()
control_counts = atlas.layers["counts"][is_control]
control_dataset = atlas.obs.loc[is_control, "dataset"].astype(str).to_numpy()

is_excluded = atlas.var["mito"] | atlas.var["transgene"] | atlas.var["symbol"].isin(EXCLUDED_GENES)
is_excluded = is_excluded.to_numpy()

cells_detected = control_counts.getnnz(axis=0)

datasets_detected = np.zeros(atlas.n_vars, dtype=int)

for dataset in np.unique(control_dataset):
    cells_detected_in_dataset = control_counts[control_dataset == dataset].getnnz(axis=0)
    datasets_detected += cells_detected_in_dataset >= MIN_CELLS_PER_DATASET

is_eligible = ~is_excluded & (cells_detected >= MIN_CONTROL_CELLS) & (datasets_detected >= MIN_DATASETS)

print(f"{int(is_eligible.sum()):,} eligible genes")

Then scanpy's `seurat_v3` method calls the 4,000 most variable genes in each dataset separately
(`batch_key="dataset"`). It reports for every gene in how many datasets it is an HVG (`highly_variable_nbatches`)
and its rank (`highly_variable_rank`). The 2,000 genes that are HVGs in the most datasets are kept
(at least 4 datasets; ties are decided by the rank).

In [ ]:
controls = ad.AnnData(
    obs=atlas.obs.loc[is_control, ["dataset"]].copy(),
    var=pd.DataFrame(index=atlas.var_names[is_eligible]),
    layers={"counts": control_counts[:, is_eligible]},
)

sc.pp.highly_variable_genes(
    controls,
    layer="counts",
    flavor="seurat_v3",
    n_top_genes=HVGS_PER_DATASET,
    batch_key="dataset",
    span=0.3,
)

candidates = controls.var[controls.var["highly_variable_nbatches"] >= MIN_DATASETS]
candidates = candidates.sort_values(
    ["highly_variable_nbatches", "highly_variable_rank"],
    ascending=[False, True],
)
hvg_ids = candidates.index[:N_HVGS]

In [ ]:
hvg_table = pd.DataFrame(
    {
        "symbol": atlas.var.loc[hvg_ids, "symbol"],
        "n_datasets": candidates.loc[hvg_ids, "highly_variable_nbatches"].astype(int),
        "rank": candidates.loc[hvg_ids, "highly_variable_rank"],
    }
)
hvg_table.index.name = "gene_id"

HVG_TABLE.parent.mkdir(parents=True, exist_ok=True)
hvg_table.to_csv(HVG_TABLE, sep="\t")

display(hvg_table["n_datasets"].value_counts().sort_index().to_frame("HVGs"))

## 2. Circulating cells and PCA

From here on only the circulating cells are used (`compartment == "circulating"`). This leaves out the lymph
gland samples (Girard, part of the Drop-seq data) and Cattenoz (circulating and resident cells together).

PCA input: log-normalised expression of the 2,000 HVGs, each gene divided by its standard deviation
(not centred, values capped at 10). 15 principal components are kept.

In [ ]:
is_circulating = atlas.obs["compartment"].eq("circulating").to_numpy()
is_hvg = atlas.var_names.isin(hvg_ids)

cells = atlas.obs.loc[is_circulating, ["dataset", "condition", "harmony_batch", "published_label"]].copy()
display(pd.crosstab(cells["dataset"], cells["condition"], margins=True))

pca_data = ad.AnnData(X=atlas.X[is_circulating][:, is_hvg].copy(), obs=pd.DataFrame(index=cells.index))

sc.pp.scale(pca_data, zero_center=False, max_value=SCALE_MAX_VALUE)
sc.pp.pca(pca_data, n_comps=N_PCS, zero_center=True, svd_solver="arpack", random_state=SEED)

pca = np.asarray(pca_data.obsm["X_pca"], dtype=np.float32)

explained = 100 * pca_data.uns["pca"]["variance_ratio"].sum()
print(f"{len(cells):,} circulating cells; {N_PCS} PCs explain {explained:.1f}% of the HVG variance")

## 3. Harmony

Harmony moves the cells in PCA space so that the batches overlap. What counts as a batch decides what is
removed. Here the batch is the **matched control experiment** (`harmony_batch` of the sample sheet): every
perturbed experiment has the same batch name as its own control experiment (for example AML1-ETO and RasV12 share
the batch of their control arm, wasp-infested larvae that of the uninfected larvae of the same study and time point).
Differences between labs and technologies are removed, but the difference between a perturbed sample and its own
control is not treated as a batch effect.

In [ ]:
cells["harmony_batch"] = pd.Categorical(cells["harmony_batch"].astype(str))

harmony_result = harmonypy.run_harmony(
    pca,
    cells,
    "harmony_batch",
    theta=HARMONY_THETA,
    max_iter_harmony=HARMONY_MAX_ITERATIONS,
    random_state=SEED,
    verbose=False,
)

harmony = np.asarray(harmony_result.Z_corr, dtype=np.float32)

## 4. UMAP

The neighbour graph (20 neighbours) is built in the Harmony space. UMAP then draws it in two dimensions.

UMAP needs starting positions. Here every cell starts at the position it had in an earlier version of this
atlas (`umap_start_positions.tsv.gz`; that version still contained the unlabelled cells). The graph is the new
one; the starting positions only keep the overall arrangement of the picture the same. The UMAP is used for
display only: every analysis uses the Harmony coordinates.

In [ ]:
start_positions = pd.read_csv(UMAP_START_POSITIONS, sep="\t", index_col="cell_id")
start_positions = start_positions.loc[cells.index].to_numpy(dtype=np.float32)

graph = ad.AnnData(obs=pd.DataFrame(index=cells.index))
graph.obsm["X_harmony"] = harmony

sc.pp.neighbors(graph, n_neighbors=N_NEIGHBORS, use_rep="X_harmony", random_state=SEED)
sc.tl.umap(graph, min_dist=UMAP_MIN_DIST, random_state=SEED, init_pos=start_positions)

umap = np.asarray(graph.obsm["X_umap"], dtype=np.float32)

## Save

In [ ]:
integration = ad.AnnData(obs=cells)
integration.obsm["X_harmony"] = harmony
integration.obsm["X_umap"] = umap

integration.write_h5ad(INTEGRATION_FILE)

print(f"saved {INTEGRATION_FILE}: {integration.n_obs:,} cells")